# 🔧 Project 4 — ToolForge: Multi-Tool Orchestrator

**Core Concept:** Dynamic tool selection, permission management, and usage tracking

### Architecture
User Request
      │
      ▼
Tool Registry → Permission Check
      │
      ▼
Tool Selection → Execution
      │
      ▼
Usage Tracking → Result

Install

In [1]:
!pip install -q langchain langchain-groq langchain-core loguru

API Key

In [2]:
import os
os.environ["GROQ_API_KEY"] = "your Api Key"

All Setup In One Block

In [3]:
import os
import json
import time
from datetime import datetime, timezone
from loguru import logger
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import sys

logger.remove()
logger.add(sys.stdout, format="{time:HH:mm:ss} | {level} | {message}", level="DEBUG")

# ── Tool Functions ──────────────────────────────────────────
def calculator_tool(expression: str) -> str:
    try:
        # Handle percentage tip calculations
        expression_lower = expression.lower()
        if "tip" in expression_lower or "%" in expression_lower:
            import re
            numbers = re.findall(r'\d+\.?\d*', expression)
            if len(numbers) >= 2:
                percentage = float(numbers[0])
                amount = float(numbers[1])
                tip = (percentage / 100) * amount
                total = amount + tip
                return f"Tip ({percentage}% of ${amount}): ${tip:.2f} | Total with tip: ${total:.2f}"
        allowed = set("0123456789+-*/()., ")
        if not all(c in allowed for c in expression):
            cleaned = ''.join(c for c in expression if c in allowed)
            result = eval(cleaned)
            return f"Result: {result}"
        result = eval(expression)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"
def weather_tool(city: str) -> str:
    weather_db = {
        "london": {"temp": 18, "condition": "cloudy"},
        "tokyo": {"temp": 30, "condition": "humid"},
        "new york": {"temp": 25, "condition": "sunny"},
        "hyderabad": {"temp": 35, "condition": "hot"},
        "mumbai": {"temp": 32, "condition": "humid"},
        "paris": {"temp": 20, "condition": "partly cloudy"}
    }
    city_lower = city.lower().strip()
    if city_lower in weather_db:
        data = weather_db[city_lower]
        return f"Weather in {city}: {data['temp']}°C, {data['condition']}"
    return f"Weather not available for {city}"

def database_tool(query: str) -> str:
    database = {
        "users": "Table: users | Columns: id, name, email | Rows: 15420",
        "orders": "Table: orders | Columns: id, user_id, amount | Rows: 89340",
        "products": "Table: products | Columns: id, name, price | Rows: 1205",
        "revenue": "Total revenue: $2,450,000 | This month: $189,000 | Growth: 12%"
    }
    query_lower = query.lower()
    for key, value in database.items():
        if key in query_lower:
            return f"Database result: {value}"
    return f"No data found for: {query}"

def email_tool(message: str) -> str:
    parts = message.split("|")
    if len(parts) >= 2:
        to = parts[0].strip()
        subject = parts[1].strip()
        return f"Email sent to {to} | Subject: {subject} | Status: Delivered"
    return f"Email queued: {message} | Status: Pending"

def search_tool(query: str) -> str:
    knowledge = {
        "python": "Python is a high-level programming language known for simplicity.",
        "langchain": "LangChain is a framework for building LLM applications.",
        "toolforge": "ToolForge is a multi-tool orchestration system with permissions.",
        "agentic": "Agentic AI systems can plan, reason, and use tools autonomously.",
        "orchestrator": "An orchestrator manages and coordinates multiple tools or services."
    }
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower:
            return f"Search result: {value}"
    return f"No results found for: {query}"

def file_tool(operation: str) -> str:
    ops = operation.lower()
    if "read" in ops:
        return "File read: config.json | Size: 2.4KB | Last modified: today"
    elif "write" in ops:
        return "File written: output.txt | Size: 1.2KB | Status: Success"
    elif "list" in ops:
        return "Files: config.json, data.csv, output.txt, report.pdf"
    return f"File operation completed: {operation}"

# ── Tool Registry ───────────────────────────────────────────
TOOL_REGISTRY = {
    "calculator": {
        "function": calculator_tool,
        "description": "Performs mathematical calculations like tip, percentage, multiply, divide",
        "category": "math",
        "permission": "public",
        "cost_per_call": 0.001,
        "avg_latency_ms": 10
    },
    "weather": {
        "function": weather_tool,
        "description": "Gets weather information for cities like Tokyo, London, Mumbai",
        "category": "external_api",
        "permission": "public",
        "cost_per_call": 0.005,
        "avg_latency_ms": 200
    },
    "database": {
        "function": database_tool,
        "description": "Queries internal database for business data like revenue, users, orders",
        "category": "internal",
        "permission": "restricted",
        "cost_per_call": 0.002,
        "avg_latency_ms": 50
    },
    "email": {
        "function": email_tool,
        "description": "Sends emails to users or teams",
        "category": "communication",
        "permission": "restricted",
        "cost_per_call": 0.003,
        "avg_latency_ms": 100
    },
    "search": {
        "function": search_tool,
        "description": "Searches knowledge base for information about topics like Python, LangChain, AI",
        "category": "knowledge",
        "permission": "public",
        "cost_per_call": 0.002,
        "avg_latency_ms": 30
    },
    "file": {
        "function": file_tool,
        "description": "Reads and writes files in the system",
        "category": "storage",
        "permission": "restricted",
        "cost_per_call": 0.001,
        "avg_latency_ms": 20
    }
}

# ── Permission Manager ──────────────────────────────────────
class PermissionManager:
    def __init__(self):
        self.user_permissions = {
            "admin": ["public", "restricted", "admin"],
            "standard": ["public"],
            "analyst": ["public", "restricted"]
        }

    def check_permission(self, user_role: str, tool_name: str) -> bool:
        if tool_name not in TOOL_REGISTRY:
            return False
        tool_permission = TOOL_REGISTRY[tool_name]["permission"]
        allowed = self.user_permissions.get(user_role, ["public"])
        return tool_permission in allowed

    def get_allowed_tools(self, user_role: str) -> list:
        allowed = self.user_permissions.get(user_role, ["public"])
        return [
            name for name, info in TOOL_REGISTRY.items()
            if info["permission"] in allowed
        ]

# ── Usage Tracker ───────────────────────────────────────────
class UsageTracker:
    def __init__(self):
        self.usage_log = []
        self.total_cost = 0.0
        self.total_calls = 0

    def log_call(self, tool_name: str, success: bool, latency_ms: float):
        cost = TOOL_REGISTRY[tool_name]["cost_per_call"]
        self.total_cost += cost
        self.total_calls += 1
        self.usage_log.append({
            "tool": tool_name,
            "success": success,
            "latency_ms": round(latency_ms, 2),
            "cost": cost,
            "timestamp": datetime.now(timezone.utc).isoformat()
        })
        logger.info(f"Tool: {tool_name} | Success: {success} | Cost: ${cost}")

    def get_report(self) -> dict:
        return {
            "total_calls": self.total_calls,
            "total_cost": round(self.total_cost, 4),
            "usage_log": self.usage_log
        }

permission_manager = PermissionManager()
usage_tracker = UsageTracker()

# ── LLM Setup ───────────────────────────────────────────────
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    api_key=os.environ["GROQ_API_KEY"]
)

def build_prompt(user_role: str) -> ChatPromptTemplate:
    allowed_tools = permission_manager.get_allowed_tools(user_role)
    tools_info = "\n".join([
        f"- {name}: {TOOL_REGISTRY[name]['description']} (category: {TOOL_REGISTRY[name]['category']})"
        for name in allowed_tools
    ])
    tool_names = ", ".join(allowed_tools)

    system = f"""You are ToolForge, a multi-tool orchestrator.
Analyze the user request carefully and select the MOST APPROPRIATE tool.

Available tools:
{tools_info}

Examples of correct tool selection:
- "calculate", "math", "tip", "percentage", "multiply" → use calculator
- "weather", "temperature", "forecast", city names → use weather
- "search", "what is", "tell me about", "information" → use search

Respond in EXACTLY this format with no extra text:
SELECTED_TOOL: [exact tool name]
TOOL_INPUT: [actual input value not a variable name]
REASONING: [one line explanation]

Tool name must be exactly one of: {tool_names}
TOOL_INPUT must be the actual value never write placeholder text."""

    return ChatPromptTemplate.from_messages([
        ("system", system),
        ("human", "Request: {request}")
    ])

# ── Orchestrator ─────────────────────────────────────────────
def execute_tool(tool_name: str, tool_input: str) -> tuple:
    if tool_name not in TOOL_REGISTRY:
        return False, f"Tool '{tool_name}' not found"
    start_time = time.time()
    try:
        tool_fn = TOOL_REGISTRY[tool_name]["function"]
        result = tool_fn(tool_input)
        latency = (time.time() - start_time) * 1000
        usage_tracker.log_call(tool_name, True, latency)
        return True, result
    except Exception as e:
        latency = (time.time() - start_time) * 1000
        usage_tracker.log_call(tool_name, False, latency)
        return False, f"Tool execution error: {str(e)}"

def parse_llm_response(response: str) -> tuple:
    tool_name = None
    tool_input = None
    reasoning = None
    for line in response.strip().split('\n'):
        if line.startswith("SELECTED_TOOL:"):
            tool_name = line.replace("SELECTED_TOOL:", "").strip()
        elif line.startswith("TOOL_INPUT:"):
            tool_input = line.replace("TOOL_INPUT:", "").strip()
        elif line.startswith("REASONING:"):
            reasoning = line.replace("REASONING:", "").strip()
    return tool_name, tool_input, reasoning

def orchestrate(request: str, user_role: str = "standard") -> dict:
    logger.info(f"Orchestrating: {request[:50]} | Role: {user_role}")
    allowed_tools = permission_manager.get_allowed_tools(user_role)
    prompt = build_prompt(user_role)
    chain = prompt | llm
    response = chain.invoke({"request": request})
    raw_output = response.content
    logger.debug(f"LLM: {raw_output}")
    tool_name, tool_input, reasoning = parse_llm_response(raw_output)

    if not tool_name:
        return {"request": request, "success": False, "error": "Could not parse tool selection"}

    if not permission_manager.check_permission(user_role, tool_name):
        logger.warning(f"Permission denied: {user_role} cannot use {tool_name}")
        return {
            "request": request,
            "success": False,
            "error": f"Permission denied: {user_role} cannot use {tool_name}",
            "selected_tool": tool_name
        }

    success, result = execute_tool(tool_name, tool_input or "")
    return {
        "request": request,
        "user_role": user_role,
        "selected_tool": tool_name,
        "tool_input": tool_input,
        "reasoning": reasoning,
        "result": result,
        "success": success
    }

def display_result(result: dict):
    print("\n" + "="*50)
    print("TOOLFORGE RESULT")
    print("="*50)
    print(f"Request      : {result['request']}")
    print(f"User Role    : {result.get('user_role', 'N/A')}")
    print(f"Selected Tool: {result.get('selected_tool', 'N/A')}")
    print(f"Tool Input   : {result.get('tool_input', 'N/A')}")
    print(f"Reasoning    : {result.get('reasoning', 'N/A')}")
    print(f"Success      : {result['success']}")
    print(f"Result       : {result.get('result', result.get('error', 'N/A'))}")
    print("="*50)

logger.info("ToolForge fully initialized")
print("All setup complete — ready to orchestrate")

05:15:08 | INFO | ToolForge fully initialized
All setup complete — ready to orchestrate


Standard User Tests

In [4]:
print("========== STANDARD USER TESTS ==========\n")

requests_standard = [
    "Calculate 15% tip on a $85 restaurant bill",
    "What is the weather like in Tokyo?",
    "Search for information about LangChain"
]

for req in requests_standard:
    result = orchestrate(req, user_role="standard")
    display_result(result)

========== STANDARD USER TESTS ==========

05:15:08 | INFO | Orchestrating: Calculate 15% tip on a $85 restaurant bill | Role: standard
05:15:08 | DEBUG | LLM: SELECTED_TOOL: calculator
TOOL_INPUT: 15% tip on $85
REASONING: The request involves calculating a percentage, which is a mathematical operation handled by the calculator tool.
05:15:08 | INFO | Tool: calculator | Success: True | Cost: $0.001

TOOLFORGE RESULT
Request      : Calculate 15% tip on a $85 restaurant bill
User Role    : standard
Selected Tool: calculator
Tool Input   : 15% tip on $85
Reasoning    : The request involves calculating a percentage, which is a mathematical operation handled by the calculator tool.
Success      : True
Result       : Tip (15.0% of $85.0): $12.75 | Total with tip: $97.75
05:15:08 | INFO | Orchestrating: What is the weather like in Tokyo? | Role: standard
05:15:09 | DEBUG | LLM: SELECTED_TOOL: weather
TOOL_INPUT: Tokyo
REASONING: The request is about getting weather information for a specific

Admin User Tests

In [5]:
print("========== ADMIN USER TESTS ==========\n")

requests_admin = [
    "Query the database for revenue information",
    "Send an email to team@company.com | Weekly Report Ready",
    "List all files in the system"
]

for req in requests_admin:
    result = orchestrate(req, user_role="admin")
    display_result(result)

========== ADMIN USER TESTS ==========

05:15:09 | INFO | Orchestrating: Query the database for revenue information | Role: admin
05:15:10 | DEBUG | LLM: SELECTED_TOOL: database
TOOL_INPUT: revenue information
REASONING: The request is for internal business data, which matches the database tool's functionality.
05:15:10 | INFO | Tool: database | Success: True | Cost: $0.002

TOOLFORGE RESULT
Request      : Query the database for revenue information
User Role    : admin
Selected Tool: database
Tool Input   : revenue information
Reasoning    : The request is for internal business data, which matches the database tool's functionality.
Success      : True
Result       : Database result: Total revenue: $2,450,000 | This month: $189,000 | Growth: 12%
05:15:10 | INFO | Orchestrating: Send an email to team@company.com | Weekly Report  | Role: admin
05:15:10 | DEBUG | LLM: SELECTED_TOOL: email
TOOL_INPUT: team@company.com | Weekly Report Ready
REASONING: The request is to send an email, which m

Permission Denied Test

In [6]:
print("========== PERMISSION DENIED TEST ==========\n")

# Force test by directly checking permission
tool_to_test = "database"
user_role = "standard"

allowed = permission_manager.check_permission(user_role, tool_to_test)
print(f"Can '{user_role}' use '{tool_to_test}'? {allowed}")
print(f"Permission denied correctly: {not allowed}")

print(f"\nAllowed tools for '{user_role}': {permission_manager.get_allowed_tools(user_role)}")
print(f"Allowed tools for 'admin': {permission_manager.get_allowed_tools('admin')}")
print(f"Allowed tools for 'analyst': {permission_manager.get_allowed_tools('analyst')}")

print("\n--- Simulating permission denial ---")
result = {
    "request": "Query the database for user information",
    "success": False,
    "error": f"Permission denied: {user_role} cannot use {tool_to_test}",
    "selected_tool": tool_to_test,
    "user_role": user_role
}
display_result(result)
print(f"\nPermission system working: {not allowed}")

========== PERMISSION DENIED TEST ==========

Can 'standard' use 'database'? False
Permission denied correctly: True

Allowed tools for 'standard': ['calculator', 'weather', 'search']
Allowed tools for 'admin': ['calculator', 'weather', 'database', 'email', 'search', 'file']
Allowed tools for 'analyst': ['calculator', 'weather', 'database', 'email', 'search', 'file']

--- Simulating permission denial ---

TOOLFORGE RESULT
Request      : Query the database for user information
User Role    : standard
Selected Tool: database
Tool Input   : N/A
Reasoning    : N/A
Success      : False
Result       : Permission denied: standard cannot use database

Permission system working: True


Usage Report

In [7]:
print("========== USAGE REPORT ==========\n")
report = usage_tracker.get_report()
print(f"Total Calls  : {report['total_calls']}")
print(f"Total Cost   : ${report['total_cost']}")
print(f"\nDetailed Log:")
for log in report['usage_log']:
    print(f"  Tool: {log['tool']:12} | Success: {log['success']} | Latency: {log['latency_ms']}ms | Cost: ${log['cost']}")

========== USAGE REPORT ==========

Total Calls  : 6
Total Cost   : $0.014

Detailed Log:
  Tool: calculator   | Success: True | Latency: 0.15ms | Cost: $0.001
  Tool: weather      | Success: True | Latency: 0.01ms | Cost: $0.005
  Tool: search       | Success: True | Latency: 0.0ms | Cost: $0.002
  Tool: database     | Success: True | Latency: 0.01ms | Cost: $0.002
  Tool: email        | Success: True | Latency: 0.0ms | Cost: $0.003
  Tool: file         | Success: True | Latency: 0.0ms | Cost: $0.001


Project Summary

In [8]:
print("========== TOOLFORGE SUMMARY ==========\n")
print("Project      : ToolForge — Multi-Tool Orchestrator")
print("Author       : K Murali Krishna")
print("Model        : Groq LLaMA-3.3-70b-versatile")
print("\nRegistered Tools:")
for name, info in TOOL_REGISTRY.items():
    print(f"  ✓ {name:12} | {info['category']:15} | {info['permission']:10} | Cost: ${info['cost_per_call']}")
print("\nKey Capabilities:")
print("  ✓ Dynamic tool selection based on request")
print("  ✓ Role-based permission management")
print("  ✓ Usage tracking with cost monitoring")
print("  ✓ Latency tracking per tool call")
print("  ✓ Graceful permission denial handling")
print("\nProduction Concepts Demonstrated:")
print("  ✓ Tool registry pattern")
print("  ✓ Permission-based access control")
print("  ✓ Cost tracking for tool usage")
print("  ✓ Orchestration vs automation")

========== TOOLFORGE SUMMARY ==========

Project      : ToolForge — Multi-Tool Orchestrator
Author       : K Murali Krishna
Model        : Groq LLaMA-3.3-70b-versatile

Registered Tools:
  ✓ calculator   | math            | public     | Cost: $0.001
  ✓ weather      | external_api    | public     | Cost: $0.005
  ✓ database     | internal        | restricted | Cost: $0.002
  ✓ email        | communication   | restricted | Cost: $0.003
  ✓ search       | knowledge       | public     | Cost: $0.002
  ✓ file         | storage         | restricted | Cost: $0.001

Key Capabilities:
  ✓ Dynamic tool selection based on request
  ✓ Role-based permission management
  ✓ Usage tracking with cost monitoring
  ✓ Latency tracking per tool call
  ✓ Graceful permission denial handling

Production Concepts Demonstrated:
  ✓ Tool registry pattern
  ✓ Permission-based access control
  ✓ Cost tracking for tool usage
  ✓ Orchestration vs automation
